In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("C:/Users/thanh/Documents/HPC Final Term/credit_risk_predictor/data/raw/default_of_credit_card_clients.csv")
df.head(2)

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_A

**Kiểm tra Imbalance**

In [4]:
print(f"Phần trăm số nhãn là 1 trên tổng các record: {len(df[df['default payment next month'] == 1])*100/len(df)}%")

Phần trăm số nhãn là 1 trên tổng các record: 22.12%


In [5]:
df.loc[df['EDUCATION'].isin([0,4,5,6]), 'EDUCATION'] = 4
df.loc[df['MARRIAGE'].isin([0,3,4]), 'MARRIAGE'] = 3

In [6]:
df.describe()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000
mean,15000.500000,167484.322667,1.603733,1.842267,1.557267,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,...,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,0.221200
std,8660.398374,129747.661567,0.489129,0.744494,0.521405,9.217904,1.123802,1.197186,1.196868,1.169139,...,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,0.415062
min,1.000000,10000.000000,1.000000,1.000000,1.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,0.000000
25%,7500.750000,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,0.000000
50%,15000.500000,140000.000000,2.000000,2.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,...,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,0.000000
75%,22500.250000,240000.000000,2.000000,2.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,...,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,0.000000
max,30000.000000,1000000.000000,2.000000,4.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000,1.000000


**Kiểm tra outliers**

In [7]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

def detect_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers_iqr = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    z_scores = np.abs(stats.zscore(df[col].dropna()))
    outliers_z = df.iloc[np.where(z_scores > 3)]

    return outliers_iqr, outliers_z

def plot_dist(df, col):
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))

    sns.histplot(df[col], kde=True, ax=axes[0])
    axes[0].set_title(f'Histogram of {col}')

    sns.boxplot(x=df[col], ax=axes[1])
    axes[1].set_title(f'Boxplot of {col}')

    plt.show()

In [8]:
outliers_col = []
num_cols = ['LIMIT_BAL', 'AGE',
            'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
            'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
for c in num_cols:
    iqr, z = detect_outliers(df, c)
    print(f"Column {c} has {iqr.shape[0]} outliers")
    if iqr.shape[0] > 0:
        outliers_col.append(c)

Column LIMIT_BAL has 167 outliers
Column AGE has 272 outliers
Column BILL_AMT1 has 2400 outliers
Column BILL_AMT2 has 2395 outliers
Column BILL_AMT3 has 2469 outliers
Column BILL_AMT4 has 2622 outliers
Column BILL_AMT5 has 2725 outliers
Column BILL_AMT6 has 2693 outliers
Column PAY_AMT1 has 2745 outliers
Column PAY_AMT2 has 2714 outliers
Column PAY_AMT3 has 2598 outliers
Column PAY_AMT4 has 2994 outliers
Column PAY_AMT5 has 2945 outliers
Column PAY_AMT6 has 2958 outliers


*==> Giữ lại outlier, vì bỏ đi có thể gây mất mát thông tin. Bên cạnh đó, trong ngành tài chính, những người có hành vi chi tiêu hoặc nợ đột biến (outlier) chính là nhóm khách hàng mang lại rủi ro cao nhất cho ngân hàng.*
--> Xử lý những cột này bằng RobustScaler thay vì StandardScaler hay MinMaxScaler

**Xử lý dòng trùng nhau**

In [10]:
num_duplicates = df.duplicated().sum()
print(f"Có {num_duplicates} dòng bị trùng hoàn toàn.")

Có 0 dòng bị trùng hoàn toàn.


In [11]:
duplicate_rows = df[df.duplicated()]
all_duplicate_rows = df[df.duplicated(keep=False)]

In [12]:
df.describe()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000
mean,15000.500000,167484.322667,1.603733,1.842267,1.557267,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,...,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,0.221200
std,8660.398374,129747.661567,0.489129,0.744494,0.521405,9.217904,1.123802,1.197186,1.196868,1.169139,...,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,0.415062
min,1.000000,10000.000000,1.000000,1.000000,1.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,0.000000
25%,7500.750000,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,0.000000
50%,15000.500000,140000.000000,2.000000,2.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,...,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,0.000000
75%,22500.250000,240000.000000,2.000000,2.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,...,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,0.000000
max,30000.000000,1000000.000000,2.000000,4.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000,1.000000


# Tiền xử lý dữ liệu: Chuẩn hóa và Mã hóa biến phân loại

In [13]:
"""
Preprocessing Pipeline - Sklearn + Imbalanced-learn
=================================================================
Bao gồm:
  1. Encoding     : Robust & OneHot
  2. Scaling      : StandardScaler fit-only-on-train
  3. Split        : Stratified 80/20
  4. Imbalance    : SMOTE vs Random Undersampling
"""
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    RobustScaler, StandardScaler,
    OneHotEncoder, OrdinalEncoder,
)
from sklearn.decomposition import PCA

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
)

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from model.feature_engineering import CreditFeatureEngineer

NUMERIC_COLS       = num_cols
TARGET_COL         = "default payment next month"
CATEGORICAL_COLS = ['SEX', 'MARRIAGE', 'limit_category', 'age_group']
ORDINAL_COLS = ['EDUCATION', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

new_engineered_num = ['max_delay', 'avg_delay', 'count_overdue', 'avg_bill', 'bill_trend', 'payment_ratio']
OTHER_NUMERIC_COLS = NUMERIC_COLS + new_engineered_num

RANDOM_STATE  = 42
TEST_SIZE     = 0.20

# ─────────────────────────────────────────────
# 1. Stratified Split
# ─────────────────────────────────────────────
def split_data(df: pd.DataFrame):
    X = df.drop(columns=['ID',TARGET_COL], axis=1)
    y = df[TARGET_COL]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        stratify=y,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    print(f"Train: {X_train.shape}, Test: {X_test.shape}")
    print(f"Train class ratio:\n{y_train.value_counts(normalize=True).round(3)}")
    print(f"Test  class ratio:\n{y_test.value_counts(normalize=True).round(3)}")
    return X_train, X_test, y_train, y_test

# ─────────────────────────────────────────────
# 3. ColumnTransformer: Encoding + Scaling
# ─────────────────────────────────────────────

def build_preprocessor() -> Pipeline:
    col_trans = ColumnTransformer(
        transformers=[
            ("num", RobustScaler(), OTHER_NUMERIC_COLS),
            ("cat", OneHotEncoder(sparse_output=False, drop="if_binary", handle_unknown='ignore'), CATEGORICAL_COLS)
        ],
        remainder='passthrough',
        verbose_feature_names_out=False,
    )

    return Pipeline([
        ('feature_creator', CreditFeatureEngineer()),
        ('column_transformer', col_trans)
    ])


# ─────────────────────────────────────────────
# 4. Imbalance Handling: SMOTE vs Undersampling
# ─────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier

SAMPLERS: dict[str, object] = {
    "No resampling":      None,
    "SMOTE":              SMOTE(random_state=RANDOM_STATE),
    "RandomUnderSampler": RandomUnderSampler(random_state=RANDOM_STATE),
}

def select_and_apply_resampling(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    preprocessor: ColumnTransformer,
) -> tuple[np.ndarray, np.ndarray, str]:
    """"
    """
    proxy_xgb = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced'
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    print(f"\n{'Chiến lược':<25}  F1-macro")
    print("─" * 45)
    results: dict[str, np.ndarray] = {}
    for name, sampler in SAMPLERS.items():

        if hasattr(preprocessor, 'steps'):
            steps = list(preprocessor.steps)
        else:
            steps = [("pre", preprocessor)]

        if sampler is not None:
            steps.append(("sampler", sampler))
        steps.append(("clf", proxy_xgb))

        scores = cross_val_score(
            ImbPipeline(steps), X_train, y_train,
            cv=cv, scoring="f1_macro", n_jobs=-1,
        )

        results[name] = scores
        print(f"{name:<25}  {scores.mean():.4f} ± {scores.std():.4f}")

    best_name = max(results, key=lambda k: results[k].mean())
    print(f"\n✅  Chiến lược tốt nhất: {best_name}")

    X_proc = preprocessor.fit_transform(X_train, y_train)
    y_arr  = y_train.to_numpy()

    best_sampler = SAMPLERS[best_name]
    if best_sampler is not None:
        X_res, y_res = best_sampler.fit_resample(X_proc, y_arr)
        print(f"Sau resampling : {X_res.shape} | class counts: {np.bincount(y_res)}")
    else:
        X_res, y_res = X_proc, y_arr
        print(f"Không resample : {X_res.shape} | class counts: {np.bincount(y_res)}")

    return X_res, y_res, best_name

In [14]:
print("=" * 60)
print("PREPROCESSING PIPELINE")
print("=" * 60)

df_final = df.copy()
print("── 1. Stratified Split ──")
X_train, X_test, y_train, y_test = split_data(df_final)

print("\n── 2. So sánh Resampling (5-fold CV) ──")
preprocessor = build_preprocessor()
X_res, y_res, best_name = select_and_apply_resampling(X_train, y_train, preprocessor)

print("\n" + "=" * 60)
print("Hoàn tất!")

PREPROCESSING PIPELINE
── 1. Stratified Split ──
Train: (24000, 23), Test: (6000, 23)
Train class ratio:
default payment next month
0    0.779
1    0.221
Name: proportion, dtype: float64
Test  class ratio:
default payment next month
0    0.779
1    0.221
Name: proportion, dtype: float64

── 2. So sánh Resampling (5-fold CV) ──

Chiến lược                 F1-macro
─────────────────────────────────────────────
No resampling              0.7013 ± 0.0041
SMOTE                      0.7006 ± 0.0043
RandomUnderSampler         0.6825 ± 0.0054

✅  Chiến lược tốt nhất: No resampling
Không resample : (24000, 38) | class counts: [18691  5309]

Hoàn tất!


In [15]:
X_test_proc = preprocessor.transform(X_test)

In [16]:
import joblib

joblib.dump(preprocessor, 'C:/Users/thanh/Documents/HPC Final Term/credit_risk_predictor/model/preprocessor.pkl')
joblib.dump(X_test_proc,'C:/Users/thanh/Documents/HPC Final Term/credit_risk_predictor/data/processed/X_test.pkl')
joblib.dump(X_res,'C:/Users/thanh/Documents/HPC Final Term/credit_risk_predictor/data/processed/X_train.pkl')
joblib.dump(y_test.to_numpy(),'C:/Users/thanh/Documents/HPC Final Term/credit_risk_predictor/data/processed/y_test.pkl')
joblib.dump(y_res,'C:/Users/thanh/Documents/HPC Final Term/credit_risk_predictor/data/processed/y_train.pkl')


['C:/Users/thanh/Documents/HPC Final Term/credit_risk_predictor/data/processed/y_train.pkl']

In [17]:
import joblib
from model.feature_engineering import CreditFeatureEngineer

feature_names = preprocessor.get_feature_names_out()

print(f"Số lượng đặc trưng sau xử lý: {len(feature_names)}")
print("Danh sách các đặc trưng:")
print(list(feature_names))

📊 Số lượng đặc trưng sau xử lý: 38
📋 Danh sách các đặc trưng:
['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'max_delay', 'avg_delay', 'count_overdue', 'avg_bill', 'bill_trend', 'payment_ratio', 'SEX_2', 'MARRIAGE_1', 'MARRIAGE_2', 'MARRIAGE_3', 'limit_category_0.0', 'limit_category_1.0', 'limit_category_2.0', 'age_group_0.0', 'age_group_1.0', 'age_group_2.0', 'age_group_3.0', 'EDUCATION', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']


In [18]:
save_path = 'C:/Users/thanh/Documents/HPC Final Term/credit_risk_predictor/model/feature_names.pkl'
joblib.dump(feature_names, save_path)

['C:/Users/thanh/Documents/HPC Final Term/credit_risk_predictor/model/feature_names.pkl']